<a href="https://colab.research.google.com/github/GuruNathan24/Tamil-NLP/blob/master/oct25.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
from pathlib import Path

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# Setup and verify the target data directory
DATA_DIR = Path("/content/drive/MyDrive/tamil_nlp")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Google Drive successfully mounted!")
print("Target Directory:", DATA_DIR)
print("Directory contents:", list(DATA_DIR.glob("*")))

Mounted at /content/drive
Google Drive successfully mounted!
Target Directory: /content/drive/MyDrive/tamil_nlp
Directory contents: [PosixPath('/content/drive/MyDrive/tamil_nlp/Tamil_Emotions .csv'), PosixPath('/content/drive/MyDrive/tamil_nlp/TamHateX.zip'), PosixPath('/content/drive/MyDrive/tamil_nlp/1780919010531-Tamil text corpus.tar.gz'), PosixPath('/content/drive/MyDrive/tamil_nlp/DravidianCodeMix-main.zip'), PosixPath('/content/drive/MyDrive/tamil_nlp/Tamil KollyWood Films Dataset - 2018.csv'), PosixPath('/content/drive/MyDrive/tamil_nlp/text.csv.zip'), PosixPath('/content/drive/MyDrive/tamil_nlp/TamHateX'), PosixPath('/content/drive/MyDrive/tamil_nlp/DravidianCodeMix-main'), PosixPath('/content/drive/MyDrive/tamil_nlp/text.csv'), PosixPath('/content/drive/MyDrive/tamil_nlp/combined_tamil_corpus.csv'), PosixPath('/content/drive/MyDrive/tamil_nlp/tamil_mlm_100k.csv'), PosixPath('/content/drive/MyDrive/tamil_nlp/xlmr_tamil_dapt')]


In [ ]:
# Re-install the packages cleanly while leaving Colab's system-managed numpy and pandas completely untouched
!pip install -q --upgrade-strategy only-if-needed \
    datasets \
    transformers \
    accelerate \
    huggingface_hub

import os
os.kill(os.getpid(), 9)

Found existing installation: numpy 2.2.6
Uninstalling numpy-2.2.6:
  Successfully uninstalled numpy-2.2.6
Found existing installation: pandas 2.2.3
Uninstalling pandas-2.2.3:
  Successfully uninstalled pandas-2.2.3
Found existing installation: datasets 5.1.0
Uninstalling datasets-5.1.0:
  Successfully uninstalled datasets-5.1.0
Found existing installation: transformers 5.20.2
Uninstalling transformers-5.20.2:
  Successfully uninstalled transformers-5.20.2
Found existing installation: accelerate 1.15.0
Uninstalling accelerate-1.15.0:
  Successfully uninstalled accelerate-1.15.0
Found existing installation: huggingface_hub 2.3.0
Uninstalling huggingface_hub-2.3.0:
  Successfully uninstalled huggingface_hub-2.3.0
error: uninstall-no-record-file

× Cannot uninstall numpy 1.26.4
╰─> The package's contents are unknown: no RECORD file was found for numpy.

hint: The package was installed by debian. You should check if it can uninstall the package.


In [ ]:
# Run this cell to programmatically restart the Google Colab runtime
import os
os.kill(os.getpid(), 9)

In [ ]:
from datasets import load_dataset

tanglish_ds = load_dataset(
    "vishnu-n/Tanglish-Corpus-185k",
    split="train"
)

print(tanglish_ds)
print(tanglish_ds.column_names)
print(tanglish_ds[0])

tanglish_texts = [
    row["text"].strip()
    for row in tanglish_ds
    if row.get("text") and row["text"].strip()
]

print("Tanglish sentences:", len(tanglish_texts))

README.md:   0%|          | 0.00/7.59k [00:00<?, ?B/s]

tanglish_corpus.jsonl: reconstructing file:   0%|          |  0.00B / 27.9MB            

tanglish_corpus.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/185973 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'source'],
    num_rows: 185973
})
['text', 'source']
{'text': 'Idhu than indha amma voda bayam pa paathukonga. Tvk is bjp b team. But they are trying to convert everyone to christians and TVK won by evm rigging. PS: emma, account history ah hide pannittu aachum post poda vendiyathu thaane.', 'source': 'r/TamilNaduDiscussion'}
Tanglish sentences: 185973


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/tamil_nlp")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset folder:", DATA_DIR)

Mounted at /content/drive
Dataset folder: /content/drive/MyDrive/tamil_nlp


In [ ]:
import zipfile

for zip_path in DATA_DIR.rglob("*.zip"):
    extract_dir = zip_path.parent / zip_path.stem

    if not extract_dir.exists():
        print("Extracting:", zip_path.name)
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(extract_dir)

print("Extraction complete.")

Extracting: TamHateX.zip
Extracting: DravidianCodeMix-main.zip
Extracting: text.csv.zip
Extracting: DravidianCodeMix-2020.zip
Extracting: journal-dataset-paper-run.zip
Extraction complete.


In [ ]:
import json
import re
import pandas as pd
from pathlib import Path
from lxml import etree

TEXT_COLUMNS = [
    "text", "comment", "comments", "sentence",
    "content", "review", "tweet", "post",
    "clean_text", "transliteration", "message"
]

def read_text_file(path):
    try:
        return path.read_text(encoding="utf-8-sig")
    except UnicodeDecodeError:
        return path.read_text(encoding="utf-8", errors="replace")

def extract_xml_text(path):
    try:
        root = etree.parse(str(path))
        return " ".join(
            x.strip() for x in root.xpath("//text()")
            if x and x.strip()
        )
    except Exception as e:
        print(f"XML read warning: {path.name}: {e}")
        return ""

def extract_text_from_frame(df):
    # Prefer known text columns
    lookup = {str(c).lower().strip(): c for c in df.columns}

    for name in TEXT_COLUMNS:
        if name in lookup:
            return df[lookup[name]].dropna().astype(str).tolist()

    # Fallback: choose the column containing the longest average text.
    # Inspect this fallback before using it in a research dataset.
    if len(df.columns) == 0:
        return []

    scores = {}
    for col in df.columns:
        values = df[col].dropna().astype(str)
        scores[col] = values.str.len().mean() if len(values) else 0

    best_col = max(scores, key=scores.get)
    print("Fallback text column:", best_col)
    return df[best_col].dropna().astype(str).tolist()

def load_local_dataset(folder, source_name):
    folder = Path(folder)
    records = []

    if not folder.exists():
        print(f"Not found: {folder}")
        return records

    extensions = {
        ".txt", ".text", ".xml", ".csv",
        ".tsv", ".json", ".jsonl", ".ndjson"
    }

    for path in folder.rglob("*"):
        if not path.is_file() or path.suffix.lower() not in extensions:
            continue

        try:
            ext = path.suffix.lower()

            if ext in {".txt", ".text"}:
                content = read_text_file(path)
                records.extend(
                    {"text": line.strip(), "source": source_name}
                    for line in content.splitlines()
                    if line.strip()
                )

            elif ext == ".xml":
                content = extract_xml_text(path)
                if content:
                    records.append({
                        "text": content,
                        "source": source_name
                    })

            elif ext in {".csv", ".tsv"}:
                sep = "\t" if ext == ".tsv" else ","
                df = pd.read_csv(path, sep=sep, on_bad_lines="skip")
                records.extend(
                    {"text": text, "source": source_name}
                    for text in extract_text_from_frame(df)
                    if text.strip()
                )

            elif ext in {".jsonl", ".ndjson"}:
                with open(path, encoding="utf-8-sig") as f:
                    for line in f:
                        if not line.strip():
                            continue
                        obj = json.loads(line)
                        if isinstance(obj, dict):
                            for col in TEXT_COLUMNS:
                                if isinstance(obj.get(col), str):
                                    records.append({
                                        "text": obj[col],
                                        "source": source_name
                                    })
                                    break
                        elif isinstance(obj, str):
                            records.append({
                                "text": obj,
                                "source": source_name
                            })

            elif ext == ".json":
                with open(path, encoding="utf-8-sig") as f:
                    obj = json.load(f)

                if isinstance(obj, list):
                    df = pd.DataFrame(obj)
                    records.extend(
                        {"text": text, "source": source_name}
                        for text in extract_text_from_frame(df)
                        if text.strip()
                    )
                elif isinstance(obj, dict):
                    for col in TEXT_COLUMNS:
                        if isinstance(obj.get(col), list):
                            records.extend(
                                {"text": str(t), "source": source_name}
                                for t in obj[col] if str(t).strip()
                            )
                            break

        except Exception as e:
            print(f"Skipping {path.name}: {e}")

    print(f"{source_name}: {len(records):,} records")
    return records

In [ ]:
import pandas as pd
from datasets import Dataset

# Source 1: Hugging Face Tanglish corpus
records = [
    {"text": row["text"], "source": "tanglish_social"}
    for row in tanglish_ds
    if row.get("text") and row["text"].strip()
]

# Sources 2–5: manually downloaded archives
source_folders = {
    "ldcil_tamil": DATA_DIR / "ldc_tamil",
    "tamil_literature": DATA_DIR / "tamil_literature",
    "tamhatex": DATA_DIR / "tamhatex",
    "dravidiancodemix": DATA_DIR / "dravidian_codemix",
}

for source, folder in source_folders.items():
    records.extend(load_local_dataset(folder, source))

df = pd.DataFrame(records)
df["text"] = df["text"].astype(str).str.strip()

# Remove empty records and duplicates
df = df[df["text"].str.len() > 0]
df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)

# Identify Tamil Unicode text for auditing
def contains_tamil_script(text):
    return bool(re.search(r"[\u0B80-\u0BFF]", text))

df["script"] = df["text"].apply(
    lambda x: "Tamil_script" if contains_tamil_script(x)
    else "Roman_or_other"
)

print("Total unique records:", len(df))
print("\nSource distribution:")
print(df["source"].value_counts())

print("\nScript distribution:")
print(df["script"].value_counts())

df.to_csv(
    DATA_DIR / "combined_tamil_corpus.csv",
    index=False,
    encoding="utf-8-sig"
)

Not found: /content/drive/MyDrive/tamil_nlp/ldc_tamil
Not found: /content/drive/MyDrive/tamil_nlp/tamil_literature
Not found: /content/drive/MyDrive/tamil_nlp/tamhatex
Not found: /content/drive/MyDrive/tamil_nlp/dravidian_codemix
Total unique records: 185973

Source distribution:
source
tanglish_social    185973
Name: count, dtype: int64

Script distribution:
script
Roman_or_other    185973
Name: count, dtype: int64


In [ ]:
TARGET_SIZE = 100_000

groups = {
    name: group
    for name, group in df.groupby("script")
}

if len(df) <= TARGET_SIZE:
    sampled_df = df.copy()
else:
    # Aim for equal representation of both script groups.
    available_groups = list(groups.keys())
    quota = TARGET_SIZE // len(available_groups)
    pieces = []

    for name, group in groups.items():
        pieces.append(
            group.sample(
                n=min(quota, len(group)),
                random_state=42
            )
        )

    sampled_df = pd.concat(pieces, ignore_index=True)

    # Fill any remaining capacity from unused records
    remaining = df.drop(index=sampled_df.index, errors="ignore")

    # Use text-based membership to avoid duplicate selections
    remaining = df[~df["text"].isin(sampled_df["text"])]

    extra = min(TARGET_SIZE - len(sampled_df), len(remaining))

    if extra > 0:
        sampled_df = pd.concat(
            [
                sampled_df,
                remaining.sample(n=extra, random_state=42)
            ],
            ignore_index=True
        )

sampled_df = sampled_df.drop_duplicates("text").reset_index(drop=True)

sampled_df[["text"]].to_csv(
    DATA_DIR / "tamil_mlm_100k.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Final corpus size:", len(sampled_df))
print(sampled_df["script"].value_counts())
print(sampled_df["source"].value_counts())

Final corpus size: 100000
script
Roman_or_other    100000
Name: count, dtype: int64
source
tanglish_social    100000
Name: count, dtype: int64


In [ ]:
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments
)

MODEL_NAME = "FacebookAI/xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# XLM-RoBERTa supports masked language modeling
model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME)

# Convert the selected texts to a Hugging Face Dataset
mlm_dataset = Dataset.from_pandas(
    sampled_df[["text"]].reset_index(drop=True)
)

# Create train and validation splits
splits = mlm_dataset.train_test_split(
    test_size=0.05,
    seed=42
)

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=128,
        padding=False
    )

tokenized = splits.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"]
)

print(tokenized)
print("GPU available:", torch.cuda.is_available())

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

...

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] XLMRobertaForMaskedLM LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Map:   0%|          | 0/95000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 95000
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 5000
    })
})
GPU available: False


In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

print("Masked language modeling configured.")

Masked language modeling configured.


In [1]:
from pathlib import Path
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer
)

# 1. Ensure DATA_DIR is defined
if 'DATA_DIR' not in locals():
    DATA_DIR = Path("/content/drive/MyDrive/tamil_nlp")

OUTPUT_DIR = str(DATA_DIR / "xlmr_tamil_dapt")
MODEL_NAME = "FacebookAI/xlm-roberta-base"

# 2. Self-healing setup: Recreate missing variables if kernel was restarted
if 'tokenized' not in locals() or 'model' not in locals() or 'tokenizer' not in locals() or 'data_collator' not in locals():
    print("Runtime variables missing. Performing self-healing recovery step...")

    csv_path = DATA_DIR / "tamil_mlm_100k.csv"
    if not csv_path.exists():
        raise FileNotFoundError(f"Could not find {csv_path}. Please run previous cells first to create the dataset.")

    print("Loading dataset...")
    sampled_df = pd.read_csv(csv_path, encoding="utf-8-sig")
    mlm_dataset = Dataset.from_pandas(sampled_df[["text"]].reset_index(drop=True))
    splits = mlm_dataset.train_test_split(test_size=0.05, seed=42)

    print("Initializing model and tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME)

    print("Tokenizing dataset...")
    def tokenize_batch(batch):
        return tokenizer(batch["text"], truncation=True, max_length=128, padding=False)
    tokenized = splits.map(tokenize_batch, batched=True, remove_columns=["text"])

    print("Configuring data collator...")
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=True,
        mlm_probability=0.15
    )
    print("Recovery complete! Proceeding to training.")

# 3. Training setup
num_train_epochs = 3
batch_size = 8
grad_accum = 2
total_steps = (len(tokenized["train"]) * num_train_epochs) // (batch_size * grad_accum)
warmup_steps = int(total_steps * 0.06)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # Training configuration
    num_train_epochs=num_train_epochs,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=max(1, warmup_steps),

    # Adjust for your GPU
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    gradient_accumulation_steps=grad_accum,

    # Evaluation and checkpoints
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # Performance
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    report_to="none",

    # Keep training reproducible
    seed=42,
    data_seed=42
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    data_collator=data_collator,
    processing_class=tokenizer
)

train_result = trainer.train()

# Save final/best model and tokenizer
trainer.save_model(OUTPUT_DIR)
tokener = tokenizer.save_pretrained(OUTPUT_DIR)

print("Training completed.")
print("Model saved to:", OUTPUT_DIR)
print(train_result.metrics)

Runtime variables missing. Performing self-healing recovery step...


FileNotFoundError: Could not find /content/drive/MyDrive/tamil_nlp/tamil_mlm_100k.csv. Please run previous cells first to create the dataset.

In [ ]:
DATASET_NAME = "tamil_mlm_100k"

In [ ]:

# ============================================================
# XLM-RoBERTa DAPT: TRAIN ONE DATASET PER EXECUTION
# Google Colab + Google Drive
# ============================================================

!pip install -q -U datasets transformers accelerate pandas scikit-learn

from pathlib import Path
import pandas as pd
import numpy as np
import torch
import json
import re
import tarfile
import zipfile
import os

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer
)

# ------------------------------------------------------------
# 1. CONFIGURATION: CHANGE THIS FOR EACH EXPERIMENT
# ------------------------------------------------------------

DATA_DIR = Path("/content/drive/MyDrive/tamil_nlp")

DATASET_NAME = "tamil_mlm_100k"

# Options:
# "tamil_mlm_100k"
# "combined_tamil_corpus"
# "text.csv"
# "Tamil_Emotions.csv"
# "TamHateX"
# "DravidianCodeMix-main"
# "Tamil_Raw_Corpus"

MODEL_NAME = "FacebookAI/xlm-roberta-base"

MAX_LENGTH = 128
NUM_EPOCHS = 3
BATCH_SIZE = 8
GRAD_ACCUM = 2
LEARNING_RATE = 5e-5

# True: continue from the latest checkpoint in this experiment.
# False: start a fresh experiment from the original XLM-R model.
RESUME_TRAINING = True

# False: independently train each dataset from the original model.
# Change this only if you deliberately want sequential adaptation.
START_FROM_PREVIOUS_DATASET_MODEL = False

# ------------------------------------------------------------
# 2. OPTIONAL: MOUNT DRIVE IF NOT ALREADY MOUNTED
# ------------------------------------------------------------

from google.colab import drive
drive.mount("/content/drive")

assert DATA_DIR.exists(), f"Folder not found: {DATA_DIR}"

# ------------------------------------------------------------
# 3. DATASET PATHS
# ------------------------------------------------------------

DATASET_PATHS = {
    "tamil_mlm_100k":
        DATA_DIR / "tamil_mlm_100k.csv",

    "combined_tamil_corpus":
        DATA_DIR / "combined_tamil_corpus.csv",

    "text.csv":
        DATA_DIR / "text.csv",

    "Tamil_Emotions.csv":
        DATA_DIR / "Tamil_Emotions.csv",

    "TamHateX":
        DATA_DIR / "TamHateX",

    "DravidianCodeMix-main":
        DATA_DIR / "DravidianCodeMix-main",

    "Tamil_Raw_Corpus":
        DATA_DIR / "1780919010531-Tamil text corpus.tar.gz",
}

if DATASET_NAME not in DATASET_PATHS:
    raise ValueError(
        f"Unknown dataset: {DATASET_NAME}\n"
        f"Choose one of: {list(DATASET_PATHS.keys())}"
    )

SELECTED_PATH = DATASET_PATHS[DATASET_NAME]

if not SELECTED_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {SELECTED_PATH}\n"
        "Check the filename in your Google Drive."
    )

print("Selected dataset:", DATASET_NAME)
print("Selected path:", SELECTED_PATH)

# ------------------------------------------------------------
# 4. EXTRACT ARCHIVES WHEN NEEDED
# ------------------------------------------------------------

def extract_archive(path):
    path = Path(path)

    if path.is_dir():
        return path

    if path.suffix.lower() == ".zip":
        out = path.parent / path.stem
        out.mkdir(parents=True, exist_ok=True)

        with zipfile.ZipFile(path) as z:
            z.extractall(out)

        return out

    if path.name.endswith((".tar.gz", ".tgz", ".tar")):
        out = path.parent / path.name.split(".tar")[0]
        out.mkdir(parents=True, exist_ok=True)

        with tarfile.open(path, "r:*") as tar:
            # Avoid extracting files outside the destination folder.
            tar.extractall(out, filter="data")

        return out

    return path

# ------------------------------------------------------------
# 5. LOAD ONLY THE SELECTED DATASET
# ------------------------------------------------------------

TEXT_COLUMN_CANDIDATES = [
    "text", "clean_text", "comment", "comments",
    "content", "sentence", "tweet", "post",
    "message", "review", "transliteration"
]

def find_text_column(df):
    columns = {
        str(c).strip().lower(): c
        for c in df.columns
    }

    for name in TEXT_COLUMN_CANDIDATES:
        if name in columns:
            return columns[name]

    # If no known column exists, choose the longest-text column.
    # Always inspect this fallback before accepting the corpus.
    scores = {}

    for col in df.columns:
        vals = df[col].dropna().astype(str)
        scores[col] = vals.str.len().mean() if len(vals) else 0

    if not scores:
        raise ValueError("No columns found in this file.")

    best = max(scores, key=scores.get)

    print("WARNING: No standard text column found.")
    print("Automatically selected:", best)

    return best

def read_dataset_file(path):
    path = Path(path)
    ext = path.suffix.lower()

    if ext in [".csv", ".tsv"]:
        sep = "\t" if ext == ".tsv" else ","

        # Read strings to avoid corrupting IDs or text.
        df = pd.read_csv(
            path,
            sep=sep,
            dtype=str,
            encoding="utf-8-sig",
            on_bad_lines="skip"
        )

        col = find_text_column(df)

        print(f"File: {path.name}")
        print("Columns:", df.columns.tolist())
        print("Text column:", col)

        return df[col].dropna().astype(str).tolist()

    if ext in [".txt", ".text"]:
        content = path.read_text(
            encoding="utf-8",
            errors="replace"
        )

        # Each non-empty line is one record.
        return [
            line.strip()
            for line in content.splitlines()
            if line.strip()
        ]

    if ext in [".jsonl", ".ndjson"]:
        texts = []

        with open(path, encoding="utf-8-sig") as f:
            for line in f:
                if not line.strip():
                    continue

                item = json.loads(line)

                if isinstance(item, dict):
                    col = find_text_column(
                        pd.DataFrame([item])
                    )
                    value = item.get(col)

                    if value is not None:
                        texts.append(str(value))

                elif isinstance(item, str):
                    texts.append(item)

        return texts

    if ext == ".json":
        with open(path, encoding="utf-8-sig") as f:
            obj = json.load(f)

        if isinstance(obj, list):
            df = pd.DataFrame(obj)
            col = find_text_column(df)
            return df[col].dropna().astype(str).tolist()

        if isinstance(obj, dict):
            for key in TEXT_COLUMN_CANDIDATES:
                if isinstance(obj.get(key), list):
                    return [
                        str(x) for x in obj[key]
                        if x is not None
                    ]

        return []

    if ext == ".xml":
        from lxml import etree

        root = etree.parse(str(path))
        return [
            " ".join(" ".join(root.getroot().itertext()).split())
        ]

    return []

def load_selected_dataset(path):
    path = extract_archive(path)

    if path.is_file():
        return read_dataset_file(path)

    # Search only within the selected directory.
    supported = {
        ".csv", ".tsv", ".txt", ".text",
        ".json", ".jsonl", ".ndjson", ".xml"
    }

    files = [
        p for p in path.rglob("*")
        if p.is_file() and p.suffix.lower() in supported
    ]

    if not files:
        raise ValueError(
            f"No supported text files found in {path}"
        )

    print("Files discovered:", len(files))

    all_texts = []

    for file in files:
        try:
            texts = read_dataset_file(file)
            all_texts.extend(texts)
            print(f"Loaded {len(texts):,}: {file.name}")

        except Exception as e:
            print(f"Skipped {file.name}: {e}")

    return all_texts

texts = load_selected_dataset(SELECTED_PATH)

# ------------------------------------------------------------
# 6. CLEAN TEXT AND REMOVE DUPLICATES
# ------------------------------------------------------------

clean_texts = []

for text in texts:
    if not isinstance(text, str):
        continue

    text = text.strip()
    text = re.sub(r"\s+", " ", text)

    if text:
        clean_texts.append(text)

df = pd.DataFrame({"text": clean_texts})
df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)

if len(df) < 2:
    raise ValueError(
        "Too few text records were loaded. "
        "Inspect the dataset structure and text column."
    )

print("\nUnique texts loaded:", len(df))
print("\nSample records:")
print(df.head(5).to_string(index=False))

# ------------------------------------------------------------
# 7. CREATE TRAIN / VALIDATION SPLIT
# ------------------------------------------------------------

dataset = Dataset.from_pandas(df, preserve_index=False)

splits = dataset.train_test_split(
    test_size=0.05,
    seed=42
)

# ------------------------------------------------------------
# 8. SELECT MODEL CHECKPOINT
# ------------------------------------------------------------

OUTPUT_DIR = DATA_DIR / f"xlmr_dapt_{DATASET_NAME.replace(" / ", "_")}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if START_FROM_PREVIOUS_DATASET_MODEL:
    previous_dir = DATA_DIR / "xlmr_tamil_dapt"

    if not (previous_dir / "config.json").exists():
        raise FileNotFoundError(
            f"No saved model found at {previous_dir}"
        )

    CHECKPOINT_SOURCE = str(previous_dir)
else:
    CHECKPOINT_SOURCE = MODEL_NAME

tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT_SOURCE)

model = AutoModelForMaskedLM.from_pretrained(
    CHECKPOINT_SOURCE
)

# ------------------------------------------------------------
# 9. TOKENIZE
# ------------------------------------------------------------

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False
    )

tokenized = splits.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"]
)

# ------------------------------------------------------------
# 10. MASKED LANGUAGE MODELING
# ------------------------------------------------------------

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

# ------------------------------------------------------------
# 11. TRAINING ARGUMENTS
# ------------------------------------------------------------

use_cuda = torch.cuda.is_available()

total_steps = (len(tokenized["train"]) * NUM_EPOCHS) // (BATCH_SIZE * GRAD_ACCUM)
warmup_steps = int(total_steps * 0.06)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),

    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_steps=max(1, warmup_steps),

    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    fp16=use_cuda,
    logging_steps=100,
    report_to="none",

    seed=42,
    data_seed=42,

    # Use more efficient attention memory when supported.
    dataloader_num_workers=2
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    data_collator=data_collator,
    processing_class=tokenizer
)

# ------------------------------------------------------------
# 12. TRAIN THIS DATASET ONLY
# ------------------------------------------------------------

latest_checkpoint = None

if RESUME_TRAINING:
    latest_checkpoint = (
        __import__(
            "transformers.trainer_utils",
            fromlist=["get_last_checkpoint"]
        ).get_last_checkpoint(str(OUTPUT_DIR))
    )

print("\nTraining dataset:", DATASET_NAME)
print("Training records:", len(tokenized["train"]))
print("Validation records:", len(tokenized["test"]))
print("Output folder:", OUTPUT_DIR)
print("GPU available:", use_cuda)

train_result = trainer.train(
    resume_from_checkpoint=latest_checkpoint
    if latest_checkpoint else None
)

# ------------------------------------------------------------
# 13. SAVE MODEL, TOKENIZER AND RESULTS
# ------------------------------------------------------------

trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

eval_results = trainer.evaluate()

results = {
    "dataset": DATASET_NAME,
    "records": len(df),
    "train_records": len(tokenized["train"]),
    "validation_records": len(tokenized["test"]),
    "evaluation": {
        key: float(value)
        for key, value in eval_results.items()
        if isinstance(value, (int, float, np.number))
    }
}

with open(OUTPUT_DIR / "results.json", "w") as f:
    json.dump(results, f, indent=2)

print("\nTraining completed!")
print("Saved model:", OUTPUT_DIR)
print("Evaluation:", eval_results)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Selected dataset: tamil_mlm_100k
Selected path: /content/drive/MyDrive/tamil_nlp/tamil_mlm_100k.csv
File: tamil_mlm_100k.csv
Columns: ['text']
Text column: text

Unique texts loaded: 100000

Sample records:
                                                                                                                                                                                         text
Vijay oru pakka dobagur leadership quality eh avanta kediyathu govt schools na padikum pothu epdi iruntuchu same school IPA pedi irukunu na real ah pathutu iruken Vijay not a quality person
                                                                                                                       Aama na intha year video edukala , kandipa next time na share panren .
                                                                                              

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] XLMRobertaForMaskedLM LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Map:   0%|          | 0/95000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]


Training dataset: tamil_mlm_100k
Training records: 95000
Validation records: 5000
Output folder: /content/drive/MyDrive/tamil_nlp/xlmr_dapt_tamil_mlm_100k
GPU available: False


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


In [2]:
import torch

gpu_available = torch.cuda.is_available()
print(f"GPU Available: {gpu_available}")
if gpu_available:
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Total GPU Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("Currently using CPU. Please enable the GPU accelerator via Runtime -> Change runtime type.")

GPU Available: False
Currently using CPU. Please enable the GPU accelerator via Runtime -> Change runtime type.


In [2]:
import torch

gpu_available = torch.cuda.is_available()
print(f"GPU Available: {gpu_available}")
if gpu_available:
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Total GPU Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("Still using CPU. If you changed the runtime type, please ensure the session is connected and active.")

GPU Available: True
GPU Device Name: Tesla T4
Total GPU Memory: 14.56 GB


In [16]:
# Re-initialize and execute the main training pipeline on GPU with robust fallback
import torch
import os
import sys
import time
from pathlib import Path
import pandas as pd
import numpy as np
from datasets import Dataset, load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer
)

# 1. Mount Google Drive
from google.colab import drive
print("Mounting Google Drive...")
drive.mount("/content/drive", force_remount=True)

# 2. Setup and synchronize directory
DATA_DIR = Path("/content/drive/MyDrive/tamil_nlp")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Synchronizing Drive folder structure...")
for i in range(5):
    try:
        _ = os.listdir(str(DATA_DIR))
        break
    except Exception:
        pass
    time.sleep(2)

csv_path = DATA_DIR / "tamil_mlm_100k.csv"
combined_path = DATA_DIR / "combined_tamil_corpus.csv"

# 3. Dynamic Self-Healing: Rebuild datasets from Hugging Face if missing from Drive
if not csv_path.exists():
    print("tamil_mlm_100k.csv not found directly. Checking for combined_tamil_corpus.csv...")
    if not combined_path.exists():
        print("Both local datasets missing! Downloading and rebuilding from Hugging Face 'Tanglish-Corpus-185k'...")
        try:
            tanglish_ds = load_dataset("vishnu-n/Tanglish-Corpus-185k", split="train")
            df = pd.DataFrame({"text": [row["text"].strip() for row in tanglish_ds if row.get("text") and row["text"].strip()]})
            df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)
            df.to_csv(combined_path, index=False, encoding="utf-8-sig")
            print(f"Rebuilt and saved {combined_path}")
        except Exception as e:
            raise RuntimeError(f"Failed to download and rebuild dataset: {e}")

    print("Re-sampling 100k subsets from base corpus...")
    df = pd.read_csv(combined_path, encoding="utf-8-sig")
    sampled_df = df.sample(n=min(100000, len(df)), random_state=42).reset_index(drop=True)
    sampled_df[["text"]].to_csv(csv_path, index=False, encoding="utf-8-sig")
    print(f"Subset successfully recreated and saved to {csv_path}!")

print(f"Successfully verified dataset at: {csv_path}")

DATASET_NAME = "tamil_mlm_100k"
MODEL_NAME = "FacebookAI/xlm-roberta-base"
OUTPUT_DIR = DATA_DIR / f"xlmr_dapt_{DATASET_NAME}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 4. Load dataset
print("Loading dataset...")
sampled_df = pd.read_csv(csv_path, encoding="utf-8-sig")
mlm_dataset = Dataset.from_pandas(sampled_df[["text"]].reset_index(drop=True))
splits = mlm_dataset.train_test_split(test_size=0.05, seed=42)

# 5. Initialize Model & Tokenizer
print("Initializing model and tokenizer on GPU...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME)

# Tokenize
def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=128, padding=False)

tokenized = splits.map(tokenize_batch, batched=True, remove_columns=["text"])

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

# 6. Configure Trainer
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=3,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=100,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    report_to="none",
    seed=42,
    data_seed=42
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    data_collator=data_collator,
    processing_class=tokenizer
)

# 7. Start Training
print(f"Active GPU for Training: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
trainer.train()

# Save results
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
print("Training completed and model saved!")

Mounting Google Drive...
Mounted at /content/drive
Synchronizing Drive folder structure...
tamil_mlm_100k.csv not found directly. Checking for combined_tamil_corpus.csv...
Both local datasets missing! Downloading and rebuilding from Hugging Face 'Tanglish-Corpus-185k'...


README.md:   0%|          | 0.00/7.59k [00:00<?, ?B/s]

tanglish_corpus.jsonl: reconstructing file:   0%|          |  0.00B / 27.9MB            

tanglish_corpus.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/185973 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.decoder.weight', 'lm_head.decoder.bias'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training completed and model saved!


In [17]:
import math

metrics = trainer.evaluate()

eval_loss = metrics["eval_loss"]
perplexity = math.exp(eval_loss) if eval_loss < 20 else float("inf")

print("Validation loss:", eval_loss)
print("Validation perplexity:", perplexity)

Training Loss,Validation Loss,Epoch
5.970705,2.976398,3


Validation loss: 2.976397752761841
Validation perplexity: 19.617023844764248


In [19]:
from transformers import pipeline
import torch

# Convert Path object to string to avoid AttributeError on tokenizer
model_path = str(OUTPUT_DIR)

fill_mask = pipeline(
    "fill-mask",
    model=model_path,
    tokenizer=model_path,
    device=0 if torch.cuda.is_available() else -1
)

# Test Tamil Unicode
print(fill_mask("இன்று <mask> மிகவும் மகிழ்ச்சியாக இருக்கிறேன்"))

# Test Tanglish
print(fill_mask("Inniku romba <mask> ah iruken"))

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[{'score': 0.05805510655045509, 'token': 1202, 'token_str': 'video', 'sequence': 'இன்று video மிகவும் மகிழ்ச்சியாக இருக்கிறேன்'}, {'score': 0.026172228157520294, 'token': 7155, 'token_str': 'bro', 'sequence': 'இன்று bro மிகவும் மகிழ்ச்சியாக இருக்கிறேன்'}, {'score': 0.02102099359035492, 'token': 319, 'token_str': 'ku', 'sequence': 'இன்றுku மிகவும் மகிழ்ச்சியாக இருக்கிறேன்'}, {'score': 0.018652789294719696, 'token': 77250, 'token_str': 'akku', 'sequence': 'இன்றுakku மிகவும் மகிழ்ச்சியாக இருக்கிறேன்'}, {'score': 0.014836141839623451, 'token': 8805, 'token_str': 'thu', 'sequence': 'இன்றுthu மிகவும் மகிழ்ச்சியாக இருக்கிறேன்'}]
[{'score': 0.6426649689674377, 'token': 17723, 'token_str': 'happy', 'sequence': 'Inniku romba happy ah iruken'}, {'score': 0.034353263676166534, 'token': 168661, 'token_str': 'tired', 'sequence': 'Inniku romba tired ah iruken'}, {'score': 0.02659774012863636, 'token': 72399, 'token_str': 'late', 'sequence': 'Inniku romba late ah iruken'}, {'score': 0.0227114930748939